In [ ]:
%pip install python-dotenv langchain-groq langgraph
from langchain_core.messages import HumanMessage, ToolMessage
from langchain.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
import os

load_dotenv()

if os.getenv("GROQ_API_KEY"):
    print("Groq API key is set.")
else:
    raise ValueError("Groq API key is not set.")
from langchain_groq import ChatGroq

model = ChatGroq(
    model="llama-3.3-70b-versatile"
)
model

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

@tool
def search_duckduckgo(query: str) -> str:
    """This tool searches the latest news on DuckDuckGo for the given query"""
    duck_search=DuckDuckGoSearchRun()
    return duck_search.invoke(query)



In [ ]:
from langchain_community.tools import ArxivQueryRun
from langchain_community.utilities import ArxivAPIWrapper
@tool

def arxiv_tool(query: str) -> str:
    """This tool searches the latest research papers on arXiv for the given query"""
    arxiv_query=ArxivQueryRun(api_wrapper=ArxivAPIWrapper())
    return arxiv_query.invoke(query)    

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

@tool
def wiki_tool(query: str):

    """This tool allows you to search Wikipedia for information on a given topic."""
    wiki_query = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
    return wiki_query.invoke(query)

In [ ]:
@tool
def personal_info(name: str):

    """Use this tool to get personal information about Alice, Bob, or Charlie. 
    """

    info = {
        "Alice": "Alice is a software engineer with 5 years of experience in AI.",
        "Bob": "Bob is a data scientist who loves working with large datasets.",
        "Charlie": "Charlie is a product manager with a background in tech startups."
    }
    return info.get(name, "No information available for this person.")

In [ ]:
tools=[search_duckduckgo, arxiv_tool, wiki_tool, personal_info]
llm_with_tools=model.bind_tools(tools)

In [ ]:
from typing import TypedDict, List

class graph_schema(TypedDict):
    messages:List

In [ ]:
def llm_node(state: graph_schema)->graph_schema:
    message=state['messages']
    
    prompt=ChatPromptTemplate.from_messages(
        [
            ("system", "You are a helpful assistant that can use tools to answer questions."),
            ("human", "{input}")
        ]
    )
    
    chain=prompt | llm_with_tools
    response=chain.invoke({"input": message})
    
    state['messages']=message + [response]
    return state

In [ ]:
def tool_node(state: graph_schema)->graph_schema:
    message=state['messages']
    tools_by_name={tool.name: tool for tool in tools}
    tool_result=[]
    for tool_call in message[-1].tool_calls:
       tool=tools_by_name[tool_call["name"]]
       observation=tool.invoke(tool_call["args"])
       tool_result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    state['messages']=message + tool_result
    return state

In [ ]:

def if_tool_call(state: graph_schema) -> str:

    last_message = state['messages'][-1]

    if last_message.tool_calls:
        return "tool_node"
    else:
        return "end"

In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(graph_schema)

# Add nodes to the graph
graph.add_node("llm_node", llm_node)
graph.add_node("tool_node", tool_node)

# Add edges between nodes
graph.add_edge(START, "llm_node")
graph.add_conditional_edges("llm_node", if_tool_call,{"tool_node": "tool_node", "end": END})
graph.add_edge("tool_node", "llm_node")

react_graph = graph.compile()

from IPython.display import Image, display

# You could see the errors with the below command
Image(react_graph.get_graph().draw_mermaid_png())

In [ ]:
react_graph.invoke({"messages": [HumanMessage(content="What is the latest news on AI?")]})